In [1]:
# a bigram character-level language model adapted from karpathy's zero to hero (makemore) lectures
# - syllabus: https://karpathy.ai/zero-to-hero.html
# - lecture: https://www.youtube.com/watch?v=PaCmpygFfXo
# - notebook: https://github.com/karpathy/nn-zero-to-hero/blob/master/lectures/makemore/makemore_part1_bigrams.ipynb
# - makemore: https://github.com/karpathy/makemore/blob/master/makemore.py#L399

print("\n\n--- DATA ---")
import torch
import torch.nn.functional as F
dataset = open('./data/names.txt', 'r').read().splitlines()
D = len(dataset)                                                               # D is the length of the dataset. N is reserved for the number of examples, in which each di can comprise many of
vocab = sorted(list(set(''.join(dataset)))) # construct vocab
c2i = {c:i+1 for i,c in enumerate(vocab)}                      # construct map<char,usize>
c2i['.'] = 0                                                                            # with . as the start token and end token, to remove counting freq of (<E>*) and (*<S>) which are all 0
V = len(c2i)                                                                   # evaluate the vocab len V

xindicesraw, yindicesraw = [], []
for di in dataset:                                                                      # change to dataset[:1] when debugging to limit the number of N examples
  di_normalized = ['.'] + list(di) + ['.']                          # normalize each word di
  for x_char,y_char in zip(di_normalized, di_normalized[1:]):               # loop through the (x,y) "self-supervised" bigrams of each word di with zip
    x_index, y_index = c2i[x_char], c2i[y_char]                               # use map<char, usize> to map representation from discrete characters to discrete integers (ords)
    xindicesraw.append(x_index), yindicesraw.append(y_index)              # append

N = len(xindicesraw)
xindices_N, yindices_N = torch.tensor(xindicesraw), torch.tensor(yindicesraw)   # then, convert python lists into torch tensors
print(f'inputs (usize): {xindices_N}'); print(f'outputs (usize): {yindices_N}')

# finally, map representation once again from discrete integers to continuous (but local) one hot vectors
x1hots_NV, y1hots_NV = F.one_hot(xindices_N,num_classes=V).float(), F.one_hot(yindices_N,num_classes=V).float()
print(f'inputs (1hot): {x1hots_NV.shape} {x1hots_NV}'); print(f'outputs (1hot): {x1hots_NV.shape} {x1hots_NV}')



--- DATA ---
inputs (usize): tensor([ 0,  5, 13,  ..., 25, 26, 24])
outputs (usize): tensor([ 5, 13, 13,  ..., 26, 24,  0])
inputs (1hot): torch.Size([228146, 27]) tensor([[1., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 1., 0.],
        [0., 0., 0.,  ..., 0., 0., 1.],
        [0., 0., 0.,  ..., 1., 0., 0.]])
outputs (1hot): torch.Size([228146, 27]) tensor([[1., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 1., 0.],
        [0., 0., 0.,  ..., 0., 0., 1.],
        [0., 0., 0.,  ..., 1., 0., 0.]])


In [2]:
print("\n\n--- ARCH ---")
g = torch.Generator().manual_seed(1337+1)                 # avgnll: 3.5 -> 4.9
W_VV = torch.randn((V, V), generator=g, requires_grad=True)  # V neurons



--- ARCH ---


In [ ]:
print("\n\n--- TRAINING ---")
K = 100
lr = 50
print(f'{D=}, {N=}, {K=}')
for k in range(K): # gradient descent
  # TRAINING FORWARD {k} --- (including softmax) with N examples from dataset D')
  logits_NV = x1hots_NV @ W_VV # batch matmul print(f'logits_NV: {logits_NV.shape}, {logits_NV}');
  counts_NV = logits_NV.exp() # equivalent to C_VV print(f'counts_NV: {counts_NV.shape}, {counts_NV}')
  probsycondx_NV = counts_NV / counts_NV.sum(dim=1, keepdims=True) # normalize, completing the evaluation of softmax
  # print(f'(forward) probsycondx_NV: {probsycondx_NV.shape}\n {probsycondx_NV}')

  # vectorized evaluation of loss in TEST section below
  indices_N = torch.arange(N) # in order to evaluate loss of first N examples, use torch.arange(N) NOT xindices_N
  loss = -probsycondx_NV[indices_N, yindices_N].log().mean() # however we do use yindices_N to find the corresponding likelihood predictions of the *targets*
  print(f'{k=}, {loss=}') # pluck the likelihoods, then eval the log, average it, and take the inverse
  # ...to get negative log likelihood

  # TRAINING BACKWARD {k} --- (including softmax) with N examples from dataset D')
  W_VV.grad = None # same as zeroing gradients
  loss.backward() # eval f'(x)
  # print(f'(backward): {W_VV.grad.shape}, {W_VV.grad}')

  # TRAINING STEP {k} ---\n')
  W_VV.data += -lr*W_VV.grad



--- TRAINING ---
D=32033, N=228146, K=100
k=0, loss=tensor(3.8281, grad_fn=<NegBackward0>)
k=1, loss=tensor(3.4267, grad_fn=<NegBackward0>)
k=2, loss=tensor(3.1815, grad_fn=<NegBackward0>)
k=3, loss=tensor(3.0264, grad_fn=<NegBackward0>)
k=4, loss=tensor(2.9231, grad_fn=<NegBackward0>)
k=5, loss=tensor(2.8483, grad_fn=<NegBackward0>)
k=6, loss=tensor(2.7928, grad_fn=<NegBackward0>)
k=7, loss=tensor(2.7508, grad_fn=<NegBackward0>)
k=8, loss=tensor(2.7180, grad_fn=<NegBackward0>)
k=9, loss=tensor(2.6916, grad_fn=<NegBackward0>)
k=10, loss=tensor(2.6698, grad_fn=<NegBackward0>)
k=11, loss=tensor(2.6515, grad_fn=<NegBackward0>)
k=12, loss=tensor(2.6359, grad_fn=<NegBackward0>)
k=13, loss=tensor(2.6225, grad_fn=<NegBackward0>)
k=14, loss=tensor(2.6107, grad_fn=<NegBackward0>)
k=15, loss=tensor(2.6004, grad_fn=<NegBackward0>)
k=16, loss=tensor(2.5913, grad_fn=<NegBackward0>)
k=17, loss=tensor(2.5831, grad_fn=<NegBackward0>)
k=18, loss=tensor(2.5759, grad_fn=<NegBackward0>)
k=19, loss=tenso

In [4]:
print("\n\n--- INFERENCE ---")





print("\n\n--- TEST ---")
i2c = {i:c for c,i in c2i.items()}                               # invert map<char, ord> to map<ord, char> for decoding

# logpD,n = 0.0, 0
# nlls = torch.zeros(N)                                            # replace sum and count tracking for average with .mean() (we won't be pushing logpycondx though only -logpycondx)
# for i in range(N):                                               # loop through the 5 input-outputs (x^(i), y^(i)) of the first word di in dataset
#   xord, yord = xindices_N[i].item(), yindices_N[i].item()        # map (x^(i), y^(i))'s index to ordinal
#   xchar, ychar = i2c[xord], i2c[yord]                            # map (x^(i), y^(i))'s ordinal to char

#   pYcondx_V = probsycondx_NV[i]
#   pycondx_ = pYcondx_V[yord]
#   logpycondx_ = torch.log(pycondx_)
#   nll = -logpycondx_
#   print(f'(x(^{i}),y(^{i})): ({xchar},{ychar}) ---> py(^{i})condx(^{i})hat: {pycondx_:.4f}, logpy(^{i})condx(^{i}): {logpycondx_:.4f}, nll: {nll:.4f}')

#   nlls[i] = nll
  # logpD += logpycondx
  # n += 1

# nllD = nlls.sum()
# avgnllD = nlls.mean()
# print(f'{nllD=}, {avgnllD=}')



--- INFERENCE ---


--- TEST ---
